In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor


from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostRegressor
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics
!pip install catboost
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from sklearn.preprocessing import StandardScaler


import matplotlib.pyplot as plt
import seaborn as sns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 99.2/99.2 MB 69.8 MB/s eta 0:00:01
ERROR: Operation cancelled by user


ModuleNotFoundError: No module named 'catboost'

In [ ]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("mexwell/boat-price-prediction") + "/Boats_Cleaned_dataset.csv"

print("Path to dataset files:", path)

# Read the CSV file using the defined path
df = pd.read_csv(path)
df.head()

#### Data Preprocessing

In [ ]:
df.drop(['Unnamed: 0', 'id',  'model', 'fuelType'], axis=1, inplace=True)


In [ ]:
# 1. Drop the unwanted columns
cols_to_drop = [
    'engineCategory', 'zip',       # object cols
    'maxEngineYear', 'minEngineYear', 'sellerId'  # numeric cols
]
df = df.drop(columns=cols_to_drop)

# 2. Remove any rows where city is missing
df = df.dropna(subset=['city', 'beam_ft', 'dryWeight_lb'])

# 3. Impute the remaining object‑type columns with their mode
# for col in ['model', 'fuelType']:
#     mode_val = df[col].mode()[0]
#     df[col].fillna(mode_val, inplace=True)

# 4. Impute the remaining numeric columns with their median
for col in [ 'totalHP']:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

# 5. (Optional) Verify that no more missing values remain in those columns
print(df[['city']].isnull().sum())


In [ ]:

# 1. Low‐cardinality → One‐hot
# --------------------------------
# type (3 categories)
d = pd.get_dummies(df['type'], prefix='type')
df[d.columns] = d
df.drop('type', axis=1, inplace=True)

# condition (2 categories) → binary map
df['condition'] = df['condition'].map({'new': 1, 'used': 0})

# hullMaterial → 3 ordinal buckets (1,2,3)
hull_counts = df['hullMaterial'].value_counts()
hull_top3  = set(hull_counts.index[:3])
hull_next3 = set(hull_counts.index[3:6])
# the remaining values fall into bucket 3
df['hullMaterial_code'] = df['hullMaterial'].apply(
    lambda x: 1 if x in hull_top3
              else 2 if x in hull_next3
              else 3
)
df.drop('hullMaterial', axis=1, inplace=True)

# fuelType (4 categories)
# d = pd.get_dummies(df['fuelType'], prefix='fuelType')
# df[d.columns] = d
# df.drop('fuelType', axis=1, inplace=True)


# 2. Medium‐cardinality → Top‐N + One‐hot
# ------------------------------------------
# boatClass (~72 categories): keep top‑10, lump rest as “Other”
#  boatClass → 3 ordinal buckets (1=top10, 2=next10, 3=rest)
boat_counts = df['boatClass'].value_counts()
boat_top10  = set(boat_counts.index[:10])
boat_next10 = set(boat_counts.index[10:20])
df['boatClass_code'] = df['boatClass'].apply(
    lambda x: 1 if x in boat_top10
              else 2 if x in boat_next10
              else 3
)
df.drop('boatClass', axis=1, inplace=True)


# 3. High‐cardinality → Frequency‐encode
# ------------------------------------------
for col in ['make', 'city', 'state']:
    counts = df[col].value_counts()
    top10  = set(counts.index[:10])
    next10 = set(counts.index[10:20])
    df[f'{col}_code'] = df[col].apply(
        lambda x, t10=top10, n10=next10: 1 if x in t10
                                         else 2 if x in n10
                                         else 3
    )
    df.drop(col, axis=1, inplace=True)


# 4. Datetime column → extract numerical features
# ---------------------------------------------------
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
df['created_weekday'] = df['created_date'].dt.weekday
df.drop('created_date', axis=1, inplace=True)


In [ ]:
boat_bool_cols = [c for c in df.columns if c.startswith('boatClass_')]

# 2. Convert True/False → 1/0 in place
df[boat_bool_cols] = df[boat_bool_cols].astype(int)

In [ ]:
# 1. Grab every bool column
bool_cols = df.select_dtypes(include='bool').columns

# 2. Cast them in place to 0/1
df[bool_cols] = df[bool_cols].astype(int)

# 3. (Optional) Verify
print(df[bool_cols].dtypes)
# should all now show int64 (or int8) instead of bool


In [ ]:
# assuming your DataFrame is df
object_cols  = df.select_dtypes(include='object').columns.tolist()
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

print("Object columns:", object_cols)
print("Numeric columns:", numeric_cols)


In [ ]:
# Per‑column missing counts
missing_object = df[object_cols].isnull().sum()
missing_numeric = df[numeric_cols].isnull().sum()

print("Missing in object columns:")
print(missing_object)

print("\nMissing in numeric columns:")
print(missing_numeric)

# Total missing across each group
total_missing_object  = missing_object.sum()
total_missing_numeric = missing_numeric.sum()

print(f"\nTotal missing values in object columns:  {total_missing_object}")
print(f"Total missing values in numeric columns: {total_missing_numeric}")


In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.head(10)

In [ ]:
correlation_matrix = df.corr(numeric_only=True)

In [ ]:
correlation_with_target = correlation_matrix['price'].sort_values(ascending=False)
print(correlation_with_target)


## Train Test Split

In [ ]:
# Features: every column except 'price'
X = df.drop(columns='price')
# X = df[['dryWeight_lb', 'length_ft', 'totalHP', 'beam_ft', 'numEngines', 'boatClass_code', 'make_code', 'year', 'condition']]


# If you need a NumPy array:
# X = df.drop(columns='price').values

# Target: the 'price' column, reshaped to a column vector
y = df['price'].values.reshape(-1, 1)


In [ ]:
X.head()

In [ ]:
X.dtypes

In [ ]:
y.dtype

In [ ]:
y.shape

In [ ]:
len(X)

In [ ]:
X.shape


In [ ]:
X.isna().sum()

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 80% train, 20% test
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42) # 60% train, 20% val

In [ ]:
# 2) Scale FEATURES (X)
X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_val_scaled   = X_scaler.transform(X_val)
X_test_scaled  = X_scaler.transform(X_test)

# 3) Scale TARGET (y) – so the network sees zero‐mean, unit‐variance prices
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled   = y_scaler.transform(y_val)
y_test_scaled  = y_scaler.transform(y_test)


## GridSearchCV

In [ ]:
model_params = {
    #  'mlp': {
    #      'model': RandomForestRegressor(),
    #      'params': {
    #          'n_estimators': [50, 100, 150, 200]
    #      }
    #  },

    'linear_regression': {
        'model': LinearRegression(),
        'params': {
            'fit_intercept': [True, False],
        }
    },




     'decision tree': {
         'model': DecisionTreeRegressor(),
         'params': {
             'max_depth': [5, 10, 15, 20]
     }
     }
    # 'knr': {
    #     'model': KNeighborsRegressor(),
    #     'params': {
    #         'n_neighbors': [1, 5, 10, 15, 20]
    #     }
    # },
    #  'catboost':{
    #     'model': CatBoostRegressor(verbose=0),
    #     'params':{
    #         'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    #         'iterations': [100, 200, 300, 400, 500],
    #      }
    #  },
    #  'svr': {
    #      'model': SVR(),
    #      'params': {
    #          'C': [0.1, 1, 10, 100, 1000],
    #          'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    #      }
    #      },
    # 'xgboost': {
    #     'model': XGBRegressor(random_state=42, verbosity=0),
    #     'params': {
    #         'n_estimators': [100, 200],
    #         'learning_rate': [0.05, 0.1],
    #     }
    # },
    # 'adaboost': {
    #     'model': AdaBoostRegressor(random_state=42),
    #     'params': {
    #         'n_estimators': [50, 100, 200],
    #         'learning_rate': [0.01, 0.05, 0.1, 0.2]
    #     }
    # },
    # 'mlp': {
    #     'model':MLPRegressor(),
    #     'params': {
    #         'n_estimators': [1, 5, 10]
    #     }
     }

In [ ]:
scores = []

for model_name, mp in model_params.items():
  print(mp['model'])
  clf2 = GridSearchCV(mp['model'], mp['params'], scoring='neg_mean_absolute_error', cv = 5, return_train_score = False)
  clf2.fit(X_train_scaled, y_train_scaled)

  scores.append({
      'model': model_name,
      'best_score': clf2.best_score_,
      'best_params': clf2.best_params_
  })

In [ ]:
scores_df = pd.DataFrame(scores, columns=['model', 'best_score', 'best_params'])
scores_df

## Best Linear & Boosting Models

In [ ]:
model_lr = LinearRegression(fit_intercept=False)
model_lr.fit(X_train_scaled, y_train_scaled)
y_pred_lr = model_lr.predict(X_test_scaled)
r2_lr = r2_score(y_test_scaled, y_pred_lr)
mae_lr = mean_absolute_error(y_test_scaled, y_pred_lr)
mse_lr = mean_squared_error(y_test_scaled, y_pred_lr)
print(f"LinearRegression R²: {r2_lr:.4f}")
print(f"LinearRegression MAE: {mae_lr:.4f}")
print(f"LinearRegression MSE: {mse_lr:.4f}")

In [ ]:
model_dt = DecisionTreeRegressor(max_depth=15)
model_dt.fit(X_train_scaled, y_train_scaled)
y_pred_dt = model_dt.predict(X_test_scaled)
r2_dt = r2_score(y_test_scaled, y_pred_dt)
mae_dt = mean_absolute_error(y_test_scaled, y_pred_dt)
mse_dt = mean_squared_error(y_test_scaled, y_pred_dt)
print(f"DecisionTreeRegressor R²: {r2_dt:.4f}")
print(f"DecisionTreeRegressor MAE: {mae_dt:.4f}")
print(f"DecisionTreeRegressor MSE: {mse_dt:.4f}")

In [ ]:
model_rf = RandomForestRegressor(n_estimators=150)
model_rf.fit(X_train_scaled, y_train_scaled)
y_pred_rf = model_rf.predict(X_test_scaled)
r2_rf = r2_score(y_test_scaled, y_pred_rf)
print(f"RandomForestRegressor R²: {r2_rf:.4f}")

In [ ]:
best_knr = KNeighborsRegressor(n_neighbors=5)
best_knr.fit(X_train_scaled, y_train_scaled)
y_pred_knr = best_knr.predict(X_test_scaled)
r2_knr = r2_score(y_test_scaled, y_pred_knr)
print(f"KNeighborsRegressor R²: {r2_knr:.4f}")

In [ ]:
best_catboost = CatBoostRegressor(learning_rate=0.1, iterations=500)
best_catboost.fit(X_train_scaled, y_train_scaled)
y_pred_catboost = best_catboost.predict(X_test_scaled)
r2_catboost = r2_score(y_test_scaled, y_pred_catboost)
print(f"CatBoostRegressor R²: {r2_catboost:.4f}")

In [ ]:
best_xgboost = XGBRegressor(learning_rate=0.1, n_estimators=200, max_depth=4)
best_xgboost.fit(X_train_scaled, y_train_scaled)
y_pred_xgb = best_xgboost.predict(X_test_scaled)
r2_xgb = r2_score(y_test_scaled, y_pred_xgb)
print(f"XGBRegressor R²: {r2_xgb:.4f}")

In [ ]:
best_adaboost = AdaBoostRegressor(n_estimators=100, learning_rate=0.1)
best_adaboost.fit(X_train_scaled, y_train_scaled)
y_pred_adaboost = best_adaboost.predict(X_test_scaled)
r2_adaboost = r2_score(y_test_scaled, y_pred_adaboost)
print(f"AdaBoostRegressor R²: {r2_adaboost:.4f}")

## XGBoost

In [ ]:
# from xgboost import XGBRegressor
# from sklearn.metrics import r2_score



# Initialize and train XGBoost model
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)
# xgb_model.fit(X_train, y_train)

xgb_model.fit(X_train_scaled, y_train_scaled.ravel())  # Flatten to 1D

# Predict and evaluate

# xgb_pred = xgb_model.predict(X_test)

y_pred_scaled = xgb_model.predict(X_test_scaled)
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()



# Evaluation metrics
mae = mean_absolute_error(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)


# Print results
print(f"XGBoost MAE:  ${mae:,.2f}")
print(f"XGBoost MSE:  ${mse:,.2f}")
print(f"XGBoost RMSE: ${rmse:,.2f}")
print(f"XGBoost R² Score: {r2:.4f}")

## ADA

In [ ]:
#ADABoost

# Initialize and train AdaBoost model
# ada_model = AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

ada_model = AdaBoostRegressor(
    #base_estimator=base_estimator,
    n_estimators=300,             # more trees
    learning_rate=0.05,           # smaller learning rate for better generalization
    loss='square',                # more sensitive loss function
    random_state=42
)

ada_model.fit(X_train_scaled, y_train_scaled.ravel())  # Flatten to 1D

# Predict on scaled X_test
y_pred_scaled = ada_model.predict(X_test_scaled)

# Inverse-transform predictions to original scale
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

# Evaluation metrics
mae = mean_absolute_error(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_scaled, y_pred_scaled)

# Print results
print(f"AdaBoost MAE:  {mae:,.2f}")
print(f"AdaBoost MSE:  {mse:,.2f}")
print(f"AdaBoost RMSE: {rmse:,.2f}")
print(f"AdaBoost R² Score: {r2:.4f}")

# CatBoost


In [ ]:
# cat_model = CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, verbose=0, random_state=42)
cat_model = CatBoostRegressor(iterations=500, learning_rate=0.1)
cat_model.fit(X_train_scaled, y_train_scaled.ravel())  # Flatten to 1D

# Predict on scaled X_test
y_pred_scaled = cat_model.predict(X_test_scaled)

# Inverse-transform predictions to original scale
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

# Evaluation metrics
mae = mean_absolute_error(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_scaled, y_pred_scaled)

# Print results
print(f"CatBoost MAE:  {mae:,.2f}")
print(f"CatBoost MSE:  {mse:,.2f}")
print(f"CatBoost RMSE: {rmse:,.2f}")
print(f"CatBoost R² Score: {r2:.4f}")


# KNN

In [ ]:
knn_model = KNeighborsRegressor(n_neighbors=2, weights='distance', p=2)

knn_model.fit(X_train_scaled, y_train_scaled.ravel())
y_pred_scaled = knn_model.predict(X_test_scaled)

y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_test_orig = y_scaler.inverse_transform(y_test_scaled).ravel()

mae = mean_absolute_error(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_scaled, y_pred_scaled)

print(f"KNN MAE:  {mae}")
print(f"KNN MSE:  {mse}")
print(f"KNN RMSE: {rmse}")
print(f"KNN R² Score: {r2}")

## MLP

In [ ]:
# 1. Define a deeper and wider network
model_mlp = models.Sequential([
    # layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    # layers.Dense(64, activation='relu'),
    # layers.Dense(32, activation='relu'),
    # layers.Dense(1)  # Output layer for regression

    layers.Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),  # Dropout for regularization
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

# 2. Compile the model with MSE loss (generally better for learning in regression)
model_mlp.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),  # You can try lower like 0.0005
    loss='mean_squared_error'
)

# 3. Train with validation split to monitor overfitting
history = model_mlp.fit(
    X_train_scaled, y_train_scaled,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)


# model_mlp.compile(optimizer=optimizers.Adam(learning_rate=0.001), loss='mean_absolute_error')
# model_mlp.fit(X_train_scaled, y_train_scaled, epochs=100, batch_size=32, verbose=1)

mae = mean_absolute_error(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_scaled, y_pred_scaled)

print(f"MAE:  ${mae:,.2f}")
print(f"MSE:  ${mse:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

## SVR

In [ ]:
svr_model = SVR(kernel='rbf', C=8000, epsilon=0.002, gamma=.0002)

svr_model.fit(X_train_scaled, y_train_scaled.ravel())

y_pred_scaled = svr_model.predict(X_test_scaled)

y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_test_orig = y_scaler.inverse_transform(y_test_scaled).ravel()

mae = mean_absolute_error(y_test_scaled, y_pred_scaled)
mse = mean_squared_error(y_test_scaled, y_pred_scaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_scaled, y_pred_scaled)

print(f"SVR MAE:  {mae}")
print(f"SVR MSE:  {mse}")
print(f"SVR RMSE: {rmse}")
print(f"SVR R² Score: {r2}")

# Plots

In [ ]:
mae_knr = mean_absolute_error(y_test, y_pred_knr)
mae_catboost = mean_absolute_error(y_test, y_pred_catboost)
mae_xgboost = mean_absolute_error(y_test, y_pred_xgb)
mae_adaboost = mean_absolute_error(y_test, y_pred_adaboost)
mae_mlp = mean_absolute_error(y_test, y_pred_mlp)
mae_svr = mean_absolute_error(y_test, y_pred_svr)

In [ ]:
mse_knr = mean_squared_error(y_test, y_pred_knr)
mse_catboost = mean_squared_error(y_test, y_pred_catboost)
mse_xgboost = mean_squared_error(y_test, y_pred_xgb)
mse_adaboost = mean_squared_error(y_test, y_pred_adaboost)
mse_mlp = mean_squared_error(y_test, y_pred_mlp)
mse_svr = mean_squared_error(y_test, y_pred_svr)